In [0]:
# Cell 1 — Bronze layer configuration
# This cell runs first every time — establishes all paths and settings

# Source path — where our raw CSVs live
source_base_path = "abfss://adventureworks@adwsadatabricksdev.dfs.core.windows.net"

# Target path — where our Bronze Delta tables will be written
# We organize Bronze tables by domain
bronze_base_path = f"{source_base_path}/bronze"

print("✅ Configuration loaded!")
print(f"   Source (CSV) : {source_base_path}")
print(f"   Target (Delta): {bronze_base_path}")
print(f"\n📁 Bronze layer folder structure:")
print(f"   {bronze_base_path}/sales/")
print(f"   {bronze_base_path}/person/")
print(f"   {bronze_base_path}/production/")

In [0]:
# Cell 2 — Read SalesTerritory from ADLS Gen2 (raw CSV)
# This is our source data — raw, unmodified, exactly as exported from SQL Server

df_territory = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{source_base_path}/salesterritory.csv")

print(f"✅ SalesTerritory loaded from CSV")
print(f"   Rows    : {df_territory.count()}")
print(f"   Columns : {len(df_territory.columns)}")
print(f"\n📋 Schema:")
df_territory.printSchema()

In [0]:
# Cell 3 — Write SalesTerritory to Bronze as a Delta table
# This is the core Bronze pattern:
# READ raw CSV → WRITE as Delta (no transformations in Bronze — raw as-is)

bronze_territory_path = f"{bronze_base_path}/sales/salesterritory"

df_territory.write \
    .format("delta")\
    .mode("overwrite")\
    .save(bronze_territory_path)


print(f"✅ SalesTerritory written to Bronze as Delta table!")
print(f"   Location : {bronze_territory_path}")
print(f"   Format   : Delta Lake")
print(f"   Mode     : Overwrite (safe for initial load)")

In [0]:
# Cell 4 — Verify: read back from Delta and confirm it matches source
# Always verify after writing — trust but verify!

df_bronze_verify = spark.read.format("delta") \
    .load(bronze_territory_path)

print(f"✅ Delta table verified!")
print(f"   Rows written : {df_bronze_verify.count()}")
print(f"   Columns      : {len(df_bronze_verify.columns)}")

# Preview first 5 rows
print(f"\n👀 Sample data from Delta table:")
df_bronze_verify.show(5, truncate=True)

In [0]:
# Cell 5 — Look inside the Delta transaction log
# This is what makes Delta different from plain Parquet
# Every write operation is recorded here

print("🔍 DELTA TRANSACTION LOG:")
print("="*50)

# List the Delta table files
delta_files = dbutils.fs.ls(bronze_territory_path)
for f in delta_files:
    print(f"   {f.name}")

print(f"\n📋 _delta_log contents:")
log_files = dbutils.fs.ls(f"{bronze_territory_path}/_delta_log/")
for f in log_files:
    print(f"   {f.name}")

In [0]:
# Cell 6 — Delta Time Travel
# Read the table at a specific version
# This works even after the data has changed!

# Read current version
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(bronze_territory_path)

print(f"✅ Time Travel — Version 0 (initial load)")
print(f"   Rows at version 0: {df_v0.count()}")

# Check full Delta history
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, bronze_territory_path)
print(f"\n📋 DELTA TABLE HISTORY:")
dt.history().select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)

In [0]:
# Cell 7 — Register the Delta table in Unity Catalog
# This makes it queryable via SQL anywhere in Databricks
# SQL equivalent: CREATE TABLE ... USING DELTA LOCATION '...'

spark.sql("""
    CREATE DATABASE IF NOT EXISTS bronze
    COMMENT 'Bronze layer - raw ingested data'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.salesterritory
    USING DELTA
    LOCATION '{bronze_territory_path}'
""")

print("✅ Table registered in Unity Catalog!")
print("   Database : bronze")
print("   Table    : salesterritory")
print("\n🔍 Verify via SQL:")

spark.sql("SELECT * FROM bronze.salesterritory LIMIT 5").show()

In [0]:
# Cell 8 — Reusable Bronze ingestion function
# Instead of repeating the same code for every table,
# we build one function that handles any table

from pyspark.sql import functions as F
from datetime import datetime

def ingest_to_bronze(table_name, csv_filename, domain, mode="overwrite"):
    """
    Reads a CSV from ADLS and writes it to Bronze as a Delta table.
    
    Args:
        table_name   : Name of the target Delta table
        csv_filename : Source CSV filename in ADLS
        domain       : Business domain (sales, person, production)
        mode         : Write mode — overwrite for full load
    """
    print(f"⏳ Ingesting {table_name}...")
    
    # Read source CSV
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{source_base_path}/{csv_filename}")
    
    # Add Bronze metadata columns
    # These columns track when and where data was loaded from
    df = df \
        .withColumn("_bronze_loaded_at", F.current_timestamp()) \
        .withColumn("_bronze_source_file", F.lit(csv_filename))
    
    # Write to Bronze Delta
    target_path = f"{bronze_base_path}/{domain}/{table_name}"
    
    df.write \
        .format("delta") \
        .mode(mode) \
        .save(target_path)
    
    # Register in Unity Catalog
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze.{table_name}
        USING DELTA
        LOCATION '{target_path}'
    """)
    
    row_count = df.count()
    print(f"✅ {table_name:<25} | Rows: {row_count:>7,} | Path: bronze/{domain}/{table_name}")
    
    return row_count

print("✅ Bronze ingestion function defined and ready!")
print("\n📌 Key design decisions:")
print("   _bronze_loaded_at   → tracks exactly when data was loaded")
print("   _bronze_source_file → tracks which source file it came from")
print("   mode=overwrite      → safe full reload every time")

In [0]:
# Cell 9 — Ingest all Sales domain tables to Bronze
# Using our reusable function — clean, consistent, no repetition

print("🚀 BRONZE INGESTION — SALES DOMAIN")
print("="*55)

# Re-define the writer here so overwrite can replace older Delta metadata
# when a table path was created earlier with a different schema.
def ingest_to_bronze(table_name, csv_filename, domain, mode="overwrite"):
    print(f"⏳ Ingesting {table_name}...")

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{source_base_path}/{csv_filename}")

    df = df \
        .withColumn("_bronze_loaded_at", F.current_timestamp()) \
        .withColumn("_bronze_source_file", F.lit(csv_filename))

    target_path = f"{bronze_base_path}/{domain}/{table_name}"

    writer = df.write \
        .format("delta") \
        .mode(mode)

    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")

    writer.save(target_path)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze.{table_name}
        USING DELTA
        LOCATION '{target_path}'
    """)

    row_count = df.count()
    print(f"✅ {table_name:<25} | Rows: {row_count:>7,} | Path: bronze/{domain}/{table_name}")

    return row_count

sales_tables = [
    ("salesorderheader",  "salesorderheader.csv",  "sales"),
    ("salesorderdetail",  "salesorderdetail.csv",   "sales"),
    ("salesterritory",    "salesterritory.csv",     "sales"),
]

total_rows = 0
start_time = datetime.now()

for table_name, csv_file, domain in sales_tables:
    rows = ingest_to_bronze(table_name, csv_file, domain)
    total_rows += rows

end_time = datetime.now()
duration = (end_time - start_time).seconds

print(f"\n{'='*55}")
print(f"✅ Sales domain ingestion complete!")
print(f"   Tables loaded : {len(sales_tables)}")
print(f"   Total rows    : {total_rows:,}")
print(f"   Duration      : {duration} seconds")

In [0]:
# Cell 10 — Verify all Sales Bronze tables
# SQL equivalent: SELECT COUNT(*) FROM each table

print("🔍 BRONZE SALES DOMAIN — VERIFICATION")
print("="*55)

for table_name, _, _ in sales_tables:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM bronze.{table_name}") \
                 .collect()[0]['cnt']
    
    # Check metadata columns were added
    cols = spark.sql(f"SELECT * FROM bronze.{table_name} LIMIT 1").columns
    has_metadata = "_bronze_loaded_at" in cols
    
    print(f"  ✅ bronze.{table_name:<25} | Rows: {count:>7,} | Metadata: {has_metadata}")

print(f"\n🎉 All Sales Bronze tables verified!")